# Data Preprocessing

This notebook prepares the real and synthetic face images for machine learning.

The preprocessing pipeline includes reproducible sampling, dataset splitting, image resizing, normalization, augmentation, and PyTorch Dataset/DataLoader construction.

## 2. Imports, repproducibility, and labels

This cell establishes the notebook-wide settings before any sampling or model training occurs.

The helper functions have three main purposes:

- set the random seed;
- create balanced and reproducibly shuffled path samples;
- calculate classification metrics without changing hidden global state.

`create_seeded_data_loader()` creates a fresh PyTorch generator every time it is called. Each neural-network experiment calls this function immediately before training.


In [ ]:
!git clone -b data-organization https://github.com/Shloka-16/deepfake-detection.git
import sys
sys.path.append('/content/deepfake-detection')

In [ ]:
# *** IMPORTS ***

import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms



Seed: 42
Label convention: Real = 0, Fake = 1
PyTorch device: cuda


In [ ]:
import sys
sys.path.append('/content/deepfake-detection')

from pathlib import Path
import torch

from src.config import SEED, REAL_LABEL, FAKE_LABEL, TRAIN_SAMPLES_PER_CLASS, VALID_SAMPLES_PER_CLASS
from src.data_paths import get_dataset_directories, list_image_paths, download_kaggle_dataset
from src.reproducibility import seed_everything, create_seeded_data_loader
from src.sampling import make_balanced_sample
from src.metrics import calculate_classification_metrics

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATASET_ROOT = Path("/content/real_vs_fake/real-vs-fake")
download_kaggle_dataset(DATASET_ROOT)

dirs = get_dataset_directories(DATASET_ROOT)

train_real_paths = list_image_paths(dirs["train_real"])
train_fake_paths = list_image_paths(dirs["train_fake"])
valid_real_paths = list_image_paths(dirs["valid_real"])
valid_fake_paths = list_image_paths(dirs["valid_fake"])

project_train_paths, project_train_targets = make_balanced_sample(
    train_real_paths, train_fake_paths,
    samples_per_class=TRAIN_SAMPLES_PER_CLASS, seed=SEED,
)
project_valid_paths, project_valid_targets = make_balanced_sample(
    valid_real_paths, valid_fake_paths,
    samples_per_class=VALID_SAMPLES_PER_CLASS, seed=SEED,
)

## 5. Create one shared train/validation sample

Every classification model will use these same image paths.

The samples come from the dataset's official training and validation folders. Seed variable controls both class sampling and the final path order.

Keeping these paths in one place prevents a PCA model, FFT model, or CNN from being evaluated on a different set of images simply because its section created a separate split.

In [ ]:
project_train_paths, project_train_targets = make_balanced_sample(
    train_real_paths,
    train_fake_paths,
    samples_per_class=TRAIN_SAMPLES_PER_CLASS,
    seed=SEED,
)

project_valid_paths, project_valid_targets = make_balanced_sample(
    valid_real_paths,
    valid_fake_paths,
    samples_per_class=VALID_SAMPLES_PER_CLASS,
    seed=SEED,
)

shared_sample_summary = pd.DataFrame(
    [
        {
            "Split": "Training",
            "Total images": len(project_train_paths),
            "Real": int((project_train_targets == REAL_LABEL).sum()),
            "Fake": int((project_train_targets == FAKE_LABEL).sum()),
        },
        {
            "Split": "Validation",
            "Total images": len(project_valid_paths),
            "Real": int((project_valid_targets == REAL_LABEL).sum()),
            "Fake": int((project_valid_targets == FAKE_LABEL).sum()),
        },
    ]
)

display(shared_sample_summary)


,Split,Total images,Real,Fake
0,Training,10000,5000,5000
1,Validation,2000,1000,1000


Training and validation paths do not overlap.


## 6. Train/Validation Separation
Training and validation images are kept separate to prevent information from the validation set from influencing training.

In [ ]:
assert len(set(project_train_paths).intersection(project_valid_paths)) == 0
print("Training and validation paths do not overlap.")

## 8. Image Preprocessing

This section defines the custom dataset class used for loading images from file paths and applies the image transforms needed for training and evaluation.

The training transform uses only modest augmentation:
- horizontal flips to reduce sensitivity to left/right orientation
- light brightness, contrast, saturation, and hue changes to improve robustness to ordinary appearance variation

Rotation was removed because it can introduce artificial corners or borders that may become shortcut features instead of meaningful facial cues. The validation and test transforms are deterministic and identical, so evaluation results are not affected by random augmentation.

In [ ]:
class FacePathDataset(Dataset):
    """Load RGB images from paths and return float binary targets."""

    def __init__(self, image_paths, numeric_targets, image_transform):
        if len(image_paths) != len(numeric_targets):
            raise ValueError("Image paths and targets must have equal length.")

        if image_transform is None:
            raise ValueError("image_transform cannot be None.")

        self.image_paths = [Path(path) for path in image_paths]
        self.numeric_targets = np.asarray(
            numeric_targets,
            dtype=np.int64,
        )
        self.image_transform = image_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[index]

        try:
            with Image.open(image_path) as image:
                rgb_image = image.convert("RGB")
        except Exception as exc:
            raise RuntimeError(
                f"Could not read image at index {index}: {image_path}"
            ) from exc

        transformed_image = self.image_transform(rgb_image)
        numeric_target = torch.tensor(
            self.numeric_targets[index],
            dtype=torch.float32,
        )

        return transformed_image, numeric_target


train_transform = transforms.Compose(
    [
        transforms.Resize(
            RESNET_IMAGE_SIZE,
            interpolation=InterpolationMode.BILINEAR,
        ),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(
            brightness=0.10,
            contrast=0.10,
            saturation=0.10,
            hue=0.02,
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=RESNET_NORMALIZE_MEAN,
            std=RESNET_NORMALIZE_STD,
        ),
    ]
)

evaluation_transform = transforms.Compose(
    [
        transforms.Resize(
            RESNET_IMAGE_SIZE,
            interpolation=InterpolationMode.BILINEAR,
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=RESNET_NORMALIZE_MEAN,
            std=RESNET_NORMALIZE_STD,
        ),
    ]
)

resnet50_train_dataset = FacePathDataset(
    image_paths=project_train_paths,
    numeric_targets=project_train_targets,
    image_transform=train_transform,
)

resnet50_valid_dataset = FacePathDataset(
    image_paths=project_valid_paths,
    numeric_targets=project_valid_targets,
    image_transform=evaluation_transform,
)

print(f"Training dataset images: {len(resnet50_train_dataset):,}")
print(f"Validation dataset images: {len(resnet50_valid_dataset):,}")

## 9. Raw RGB datasets from the shared paths

This section creates `64 × 64` RGB datasets from the same shared image paths used in the PCA and FFT experiments.

It does not create another sample or split. It also does not create the DataLoaders yet. The training cell recreates them immediately before each run.

In [ ]:
RAW_RGB_IMAGE_SIZE = (64, 64)


raw_rgb_transform = transforms.Compose(
    [
        transforms.Resize(RAW_RGB_IMAGE_SIZE),
        transforms.ToTensor(),
    ]
)

raw_rgb_train_dataset = FacePathDataset(
    project_train_paths,
    project_train_targets,
    raw_rgb_transform,
)

raw_rgb_valid_dataset = FacePathDataset(
    project_valid_paths,
    project_valid_targets,
    raw_rgb_transform,
)

print(f"Raw RGB training images: {len(raw_rgb_train_dataset)}")
print(f"Raw RGB validation images: {len(raw_rgb_valid_dataset)}")

Raw RGB training images: 10000
Raw RGB validation images: 2000


## 10. Preprocessed Image Verification

In [ ]:
# 10. Preprocessed Image Verification

import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 10.1 Verify dataset sizes
# ------------------------------------------------------------

print("Dataset sizes:")
print(f"ResNet50 training:   {len(resnet50_train_dataset):,}")
print(f"ResNet50 validation: {len(resnet50_valid_dataset):,}")
print(f"Raw RGB training:    {len(raw_rgb_train_dataset):,}")
print(f"Raw RGB validation:  {len(raw_rgb_valid_dataset):,}")


# ------------------------------------------------------------
# 10.2 Verify tensor shapes and labels
# ------------------------------------------------------------

resnet_image, resnet_label = resnet50_train_dataset[0]
raw_image, raw_label = raw_rgb_train_dataset[0]

print("\nFirst training sample:")
print(f"ResNet image shape: {resnet_image.shape}")
print(f"ResNet label:       {resnet_label}")
print(f"Raw RGB image shape: {raw_image.shape}")
print(f"Raw RGB label:       {raw_label}")


# ------------------------------------------------------------
# 10.3 Verify image value ranges
# ------------------------------------------------------------

print("\nImage value ranges:")

print(
    f"Raw RGB: "
    f"min={raw_image.min():.3f}, "
    f"max={raw_image.max():.3f}"
)

print(
    f"ResNet: "
    f"min={resnet_image.min():.3f}, "
    f"max={resnet_image.max():.3f}"
)


# ------------------------------------------------------------
# 10.4 Verify labels
# ------------------------------------------------------------

print("\nLabel values:")

print(
    "Training labels:",
    np.unique(resnet50_train_dataset.numeric_targets)
)

print(
    "Validation labels:",
    np.unique(resnet50_valid_dataset.numeric_targets)
)


# ------------------------------------------------------------
# 10.5 Display raw RGB and ResNet versions
# ------------------------------------------------------------

def denormalize_image(image_tensor, mean, std):
    """Convert a normalized tensor back to displayable [0, 1] RGB."""
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)

    image = image_tensor.cpu() * std + mean
    return image.clamp(0, 1)


resnet_display_image = denormalize_image(
    resnet_image,
    RESNET_NORMALIZE_MEAN,
    RESNET_NORMALIZE_STD,
)

raw_display_image = raw_image.cpu().clamp(0, 1)


fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].imshow(raw_display_image.permute(1, 2, 0))
axes[0].set_title(f"Raw RGB\nLabel: {int(raw_label.item())}")
axes[0].axis("off")

axes[1].imshow(resnet_display_image.permute(1, 2, 0))
axes[1].set_title(f"ResNet Preprocessed\nLabel: {int(resnet_label.item())}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Preprocessing Summary

The final pipeline:

1. Selects balanced real and fake samples.
2. Keeps training and validation images separate.
3. Resizes images to the model input dimensions.
4. Applies training augmentation where appropriate.
5. Converts images to PyTorch tensors.
6. Applies ImageNet normalization.
7. Loads the data through PyTorch DataLoaders.